# Lab 1: Statistical Machine learning - Gaussian Processes and Bayesian Optimization

In this lab, we aim to become more comfortable with Machine Learning from a probabilistic viewpoint. We will look at **kernel methods**, **fully bayesian treatment of model parameters**, **Gaussian Processes** (GPs) and **Bayesian Optimization** (BO).

The lab is divided into 4 parts.
- In **part 1**, we explore some foundations. We will work with linear regression, kernelization, and marginalization over the model parameters.
- In **part 2**, we will work with GPs, which to a large extent can be viewed as the combination of the different parts in part 1.
- In **part 3**, we  will write our own BO loop.
- Lastly, in **part 4**, we use state-of-the-art implementations to optimize "real life problem".

_Note_: Part 4 is in a separate notebook.

_Tips and tricks_:
This lab will require you to work with a lot of matrices both in numpy and in torch.
- You can use `@` to do matrix multiplication.
- When working with matrices, keeping track of matrix dimensions is essential. Make sure to keep an eye on it. Print and examine `.shape` often.
- `.reshape()` is your friend. Make sure that you understand it.

_Notes about structure_:
- The lab comes with quite some support code already written. When you need to make changes to code, it is marked with [# -> TODO: ... <-]
- Please answer the marked questions for yourselves and be ready to answer them to the TA

***
## PART 1 : Foundations
***

Let us begin with some linear regression.


In [ ]:
## import some base packages
from typing import Callable, Literal
import numpy as np
import scipy as sp
import seaborn as sns
from matplotlib import pyplot as plt

In [1]:
!wget https://raw.githubusercontent.com/ErikOrm/EDAP30-Lab1/main/winequality-white.csv
!wget https://raw.githubusercontent.com/ErikOrm/EDAP30-Lab1/main/winequality-red.csv
!wget https://raw.githubusercontent.com/ErikOrm/EDAP30-Lab1/main/blackbox_problems.py

--2026-04-01 08:31:15--  https://raw.githubusercontent.com/ErikOrm/EDAP30-Lab1/main/winequality-white.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 264402 (258K) [text/plain]
Saving to: ‘winequality-white.csv’

winequality-white.c 100%[===================>] 258.21K  --.-KB/s    in 0.005s  

2026-04-01 08:31:15 (55.9 MB/s) - ‘winequality-white.csv’ saved [264402/264402]

--2026-04-01 08:31:15--  https://raw.githubusercontent.com/ErikOrm/EDAP30-Lab1/main/winequality-red.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 10095

### Some datasets

Let us first define a data set that we can play around with.

_Note_: We already now write the data as column vectors in preparation for multi-dimensional data later.

In [ ]:
X_RANGE = [0, 1]
def gen_data(
        f_type: Literal["linear", "quadratic", "sinus", "complicated"] = "linear",
        n: int = 30,
        seed: int = 127
):
    """Generates 1 out of 4 data sets. Note that they are already normalized and standardized."""
    gen = np.random.default_rng(seed)
    X = gen.random(n) * (X_RANGE[1] - X_RANGE[0]) + X_RANGE[0]
    match f_type:
        case "linear":
            y = (
                -5 * X
                + gen.normal(0, .7, n)
            )
        case "quadratic":
            y = (
                100 * X ** 2
                + 10 *  X
                + gen.normal(0, 1, n)
            )
        case "sinus":
            y = (
                np.sin(10 * X)
                + gen.normal(0, .04, n)
            )
        case "complicated":
            y = (
                    0.8 * (X - .5) ** 3
                    + 8 * X ** 2
                    + 7 * np.sin(15 * X)
                    + 3 * X
                    + gen.normal(0, .3, n)
            )
        case _:
            raise NotImplementedError

    y = (y - y.mean()) / y.std()

    return X.reshape(-1, 1), y.reshape(-1, 1)


def plot_data(X, y, ax = None) -> None:
    """Plots data from X and y."""
    if ax is None:
        fig, ax = plt.subplots()
    ax.plot(X, y, 'o', color='red', label="data")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.legend()


X, y = gen_data(f_type = "linear")
plot_data(X, y)
plt.show()

### A basic Linear model
Let us begin with making a linear model. I have given you the scaffolds but left out the interesting parts.

**Questions**:
- how does the formula for calculating the weights looks like?
- what does the `_transform` do and why do we need it?

Go ahead and implement the model fitting method and try it on some of the data sets.

(I went ahead and added some structure for non-linear transforms for later. You can ignore them for now.)

In [ ]:
DATASET: Literal["linear", "quadratic", "sinus", "complicated"] = "linear"

def _transform(x: np.ndarray) -> np.ndarray:
    """Transforms a 1D x vector into the required 2D x vector."""
    if x.ndim == 1:
        x = x.reshape(-1, 1)
    m = np.ones_like(x)
    return np.concatenate((m, x), axis=1)


def _non_linear_transform(x: np.ndarray) -> np.ndarray:
    """Transforms a 1D x vector into rows in a feature space."""
    if x.ndim == 1:
        x = x.reshape(-1, 1)
    # ➡️ TODO (later): implement the feature mapping ⬅️
    ...


class LinearModel:

    def __init__(
            self,
            transform: Callable[[np.ndarray], np.ndarray] = _transform
    ) -> None:
        self.transform = transform
        self.weights: np.ndarray | None = None

    def predict(
        self,
        x: np.ndarray,    # 1D array with data points
    ) -> np.ndarray:
        if self.weights is None:
            raise ValueError(
                "Linear model has not been fitted yet."
            )
        # ➡️ TODO (now): complete the predict method ⬅️

    def fit(
        self,
        x: np.ndarray, # 1D array with features
        y: np.ndarray, # 1D array with values
        regularization: float = 1e-4,
    ) -> None:
        if y.ndim == 1:
            y = y.reshape(-1, 1)
        _X = self.transform(x)
        # ➡️ TODO (now): complete the fit method ⬅️
        ...




# And then we just run it on our data.
# you can change DATASET at the top of this cell to try different data
x_train, y_train = gen_data(f_type=DATASET)
x_test = np.linspace(X_RANGE[0], X_RANGE[1], 100)
model = LinearModel(_transform)
model.fit(x_train, y_train)
y_pred = model.predict(x_test)
fig, ax = plt.subplots()
plot_data(x_train, y_train, ax=ax)
ax.plot(x_test, y_pred, color="green", label='linear model')
ax.legend()
plt.show()


### Improving our Linear Predictor

Now we have a basic linear predictor. Naturally, it fails dramatically on the non-linear datasets, and as such, let us begin with making two changes to it to make it a bit more powerful.

- Add regularization (making it a ridge regressor) if you haven't already
- Add additional features:
    - Do this by implementing the `_non_linear_transform` method.
    - Examples of features could be `x^2`, `sin(cx)`, etc.

Make the changes to the model above and then run the script below (where we are using the new transform).

In [ ]:
## runs your model on all data sets

fig, axs = plt.subplots(2, 2)
x_test = np.linspace(X_RANGE[0], X_RANGE[1], 100)
for data, ax in zip(["linear", "quadratic", "sinus", "complicated"], axs.flatten()):
    x_train, y_train = gen_data(f_type=data)
    model = LinearModel(_non_linear_transform)
    model.fit(x_train, y_train, regularization=1e-6)
    y_pred = model.predict(x_test)
    plot_data(x_train, y_train, ax=ax)
    ax.plot(x_test, y_pred, color="green", label='model')
    ax.legend()
plt.show()


Alright, great! We have a functioning model. But having to manually select the features is a bit painful. For toy problems like this, the computational cost is manageable, but if you have thousands of data points and thousands or millions of features it starts to become a real issue. Let's take the next step and use the **Kernel Trick** to get Kernelized Ridge Regression.

But before we do that, let's take a detour and look at kernels.

## Kernel Functions

A **kernel function** $k(\mathbf{x}, \mathbf{x}')$ measures the *similarity* between two points:

- $k(\mathbf{x}, \mathbf{x}') \approx 1$: the two points are very similar
- $k(\mathbf{x}, \mathbf{x}') \approx 0$: the two points are very dissimilar

The most common kernel is the **Radial Basis Function (RBF)** kernel (also called the squared exponential kernel):

$$
k(\mathbf{x}, \mathbf{x}') = \exp\left(-\frac{\|\mathbf{x} - \mathbf{x}'\|^2}{2l^2}\right)
$$

The parameter $l > 0$ is the **lengthscale** — it controls how quickly similarity decays with distance. A small $l$ means only very nearby points are considered similar; a large $l$ means even distant points are treated as similar.

We know from Mercer's theorem that every kernel $k(x, x')$ corresponds to some feature mapping $\mathcal{X} \rightarrow \Phi(\mathcal{X})$, such that $k(x, x') = \Phi(x)^\top\Phi(x')$, but for the RBF, this $\Phi$ is non-trivial. If we instead look at the mapping $x \rightarrow \{x^k\}_{k=0}^{\infty}$ , then the corresponding kernel is $k(x,x') = 1/(1- xx')$.


Given $n$ training points $X = \{x_1, \ldots, x_n\}$, we assemble all pairwise similarities into a **kernel matrix** $K \in \mathbb{R}^{n \times n}$, where $K_{ij} = k(x_i, x_j)$. This is called the Gram matrix.

### Implementing the RBF Kernel

**Your task:** Implement the RBF kernel below.

The function should handle batched inputs: given two datasets $X \in \mathbb{R}^{n \times d}$ and $X' \in \mathbb{R}^{m \times d}$, return the kernel matrix $K_{XX'} \in \mathbb{R}^{n \times m}$ with the pair-wise similarities between each row in $X$ and each row in $X'$.

_Tip:_ You can compute all squared distances at once using broadcasting, or use a nested loop over pairs for clarity.

**Questions**:
- What is always on the diagonal of the Gram matrix, i.e., $K_{XX'}$ when $X = X'$? Why?

In [ ]:
def kern(X: np.ndarray, X_prime: np.ndarray, length_scale: float) -> np.ndarray:
    """
    Computes the RBF kernel matrix between X and X'.

    Args:
        X:       array of shape (n, d)
        X_prime: array of shape (m, d)
        length_scale:      lengthscale > 0

    Returns:
        K: kernel matrix of shape (n, m), where K[i, j] = k(X[i], X_prime[j])

    If X or X_prime are 1D vectors they are reshaped to (1, d) first.
    """
    if X.ndim == 1:
        X = X.reshape(1, -1)
    if X_prime.ndim == 1:
        X_prime = X_prime.reshape(1, -1)

    # ➡️ TODO: implement the RBF kernel and return the kernel matrix ⬅️

    return K

#### Safety check

The cell below visualises the kernel matrix between a grid of points and itself. You should see a matrix that is brightest on the diagonal (a point is maximally similar to itself) and decays as you move away from it.

**Question:** What does this plot tell you about the structure of the RBF kernel?

In [ ]:
GRANULARITY = 50
x_grid = np.linspace(-2, 2, GRANULARITY).reshape(-1, 1)

K = kern(x_grid, x_grid, length_scale=2)

assert K.ndim == 2, "kern must return a 2D matrix"
assert K.shape == (GRANULARITY, GRANULARITY), \
    f"Expected shape ({GRANULARITY}, {GRANULARITY}), got {K.shape}"

xlabels = ['' if i % 5 != 0 else f'{x_grid.squeeze()[i]:.1f}' for i in range(GRANULARITY)]
ylabels = ['' if i % 5 != 0 else f'{x_grid.squeeze()[i]:.1f}' for i in range(GRANULARITY)]

sns.heatmap(K, xticklabels=xlabels, yticklabels=ylabels, cmap="coolwarm")
plt.title('RBF kernel matrix (ls=2)')
plt.show()

# Kernel Ridge Regression

With a kernel in hand, we are ready to do **Kernel Ridge Regression (KRR)**.

### Derivation

Recall standard ridge regression. Given training data $(X, \mathbf{y})$, the optimal weight vector minimises the regularised least-squares objective, yielding:

$$
\mathbf{w} = (X^\top X + \lambda I)^{-1} X^\top \mathbf{y}.
$$

This is called the Normal equations. Using the **Woodbury matrix identity** we can rewrite this in its **dual form**:

$$
\mathbf{w} = X^{\top}(X X^\top + \lambda I)^{-1} \mathbf{y} \qquad \rightarrow \qquad \mathbf{w} = X^\top \boldsymbol{\alpha}, \qquad \boldsymbol{\alpha} = (X X^\top + \lambda I)^{-1} \mathbf{y}.
$$

The prediction for a new point $x^*$ is then:

$$
f(x^*) = x^* \mathbf{w} = \underbrace{x^* X^\top}_{\text{dot products with training set}} \boldsymbol{\alpha}.
$$

The quantity $x^* X^\top$ is just the *linear kernel* evaluated between $x^*$ and each training point, and $XX^\top$ is the linear kernel matrix.

The **kernel trick** says: replace every inner product the kernel function $k$:

$$
\boxed{\qquad \boldsymbol{\alpha} = (K + \lambda I)^{-1} \mathbf{y}, \qquad f(x^*) = k_* \boldsymbol{\alpha}}
$$

where:
- $K$ s.t., $K_{ij} = k(x_i, x_j)$ is the $n \times n$ training kernel matrix
- $k_* = [k(x^*, x_1), \ldots, k(x^*, x_n)] \in \mathbb{R}^{1 \times n}$ is the similarity between $x^*$ and all training points
- $\lambda > 0$ is the **regularisation strength**, preventing overfitting and ensuring the system is invertible

More information can be found in: https://www.cs.cornell.edu/courses/cs4780/2018fa/lectures/lecturenote13.html


### Your task: Implement Kernel Ridge Regression

Write the KRR model. It should:

1. Compute the $n \times n$ training kernel matrix $K$
2. Solve the linear system $(K + \lambda I)\boldsymbol{\alpha} = \mathbf{y}$ for $\boldsymbol{\alpha}$
3. Compute $k_*$, the kernel between test points and training points, shape $(m \times n)$
4. Return predictions $\hat{y} = k_* \boldsymbol{\alpha}$, shape $(m \times 1)$

**Questions**:
- Why do we need to use the kernel trick? What problem does it solve for us?
- What is the shape of $\boldsymbol{\alpha}$? What is the shape of $k_*$ when predicting $m$ test points at once?
- Why does the KRR model below not have a `fit` method?

In [ ]:
# ➡️ TODO: implement the kernel ridge regressor ⬅️
class KernelRidgeRegression:
    def __init__(
        self,
        x_train: np.ndarray,
        y_train: np.ndarray,
        length_scale: float = 1,
        regularization: float = 1e-3,
    ) -> None:
        self.x_train = x_train
        self.y_train = y_train
        self.length_scale = length_scale
        self.regularization = regularization

    def predict(
        self,
        X_test: np.ndarray,
    ) -> np.ndarray:
        ...


#### Safety check

The cell below should print `✅ Shape looks good.`

In [ ]:
x_check_train = np.random.default_rng().random((23, 33))
y_check_train = np.random.default_rng().random((23, 1))
x_check_test = np.random.default_rng().random((14, 33))
krr = KernelRidgeRegression(
    x_check_train,
    y_check_train,
    length_scale = 1,
    regularization = 1e-3
)
y_check_pred = krr.predict(x_check_test)

assert y_check_pred.shape == (x_check_test.shape[0], 1), \
    f"Expected shape ({x_check_test.shape[0]}, 1), got {y_check_pred.shape}"

print("✅ Shape looks good.")

### Running the KRR on our data

Let us visualise how well the model fits the data.

In [ ]:
## runs your model on all data sets
def plot_krr(length_scale: float, regularization: float) -> None:
    fig, axs = plt.subplots(2, 2)
    x_test = np.linspace(X_RANGE[0], X_RANGE[1], 100).reshape(-1, 1)
    for data, ax in zip(["linear", "quadratic", "sinus", "complicated"], axs.flatten()):
        x_train, y_train = gen_data(f_type=data)
        model = KernelRidgeRegression(x_train, y_train, length_scale, regularization)
        y_pred = model.predict(x_test)
        plot_data(x_train, y_train, ax=ax)
        ax.plot(x_test, y_pred, color="green", label='model')
        ax.legend()
    fig.suptitle(f"Kernel Ridge Regression - ls: {length_scale}  reg: {regularization}")

plot_krr(length_scale=.6, regularization=1e-4)
plt.show()

### Task: Experiment with hyperparameters

The cell below plots predictions for a grid of lengthscales and regularisation strengths.

**Questions:**
- What happens when the lengthscale is very **small** (e.g. `ls=0.1`)? And very **large** (e.g. `ls=10`)?
- What does a **large** $\lambda$ do to the predictions? What property does $\lambda$ trade off?
- When $\lambda \to 0$ and the kernel matrix $K$ is invertible, the predictions pass exactly through the training points. Why does this happen mathematically?
- Can you find a `length_scale` and `regularization` combination that gives a visually good fit?

In [ ]:
# ➡️ TODO: choose parameters to try here ⬅️
lengthscales = []
lambdas = []

for lam in lambdas:
    for ls in lengthscales:
        plot_krr(length_scale=ls, regularization=lam)

plt.tight_layout()
plt.show()

---
# Bayesian Linear Regression
---

Ok, so now we have flexible and efficient models, but we still lack the uncertainty estimates that GPs are so famous for. Those come from a fully Bayesian treatment of the hyperparameters.

To keep it simple, let us for a little while step back to the linear model we did at the start. We will go back and kernelize it soon enough.

When selecting the weights, so far we have been using the Normal Equations. The normal equations are achieved by either minimizing the $\mathcal{L}_2$-loss or by assuming a normal data model with fixed noise and optimize the linear weights by MLE.

Assumed data distribution:
$$p(y|x, w)\sim N(xw, \sigma^2)$$

MLE:
$$\text{arg}\max_w p(D|w)$$

MAP:
$$\text{arg}\max_w p(w|D) = \text{arg}\max_w \frac{p(D|w)p(w)}{p(D)}$$

(It is a useful exercise to show that the MLE recovers the $\mathcal{L}_2$-loss, and MAP recovers Ridge regression, but we won't require it here.)

Both MLE and MAP selects a single value for $w$. However, what we really care about is $p(y|D)$, while the weights are jsut a means to an end. And so, what we can do is that we can marginalize out the weights.

$$
p(y|D) = \int_w p(y|w, D)p(w|D)dw
=\int_w p(y|w, D)\frac{p(D|w)p(w)}{p(D)}dw,
$$
where the last step is just Bayes formula.


This is in some sense much more pure than normal linear regression, as we made away with the weights. So why don't we always do this? In almost all cases, this integral in intractable.

Fortunately, if we assume a Gaussian prior $p(w)$, all terms in the integral are gaussian and so this has a closed form solution. If we say that $D = (X_n, y_n)$, and we want to predict $y_* | x_*$.

$$p(y_*|D)\sim N(\hat{\mu}, \hat{\sigma}^2)$$
$$S_n = (X_n^\top X_n + \lambda I)$$
$$\hat{\mu} = x_*S_n^{-1} X^\top \mathbf{y_n}$$
$$\hat{\sigma}^2 = x_*S_n^{-1}x_*^\top + \sigma^2$$


Below we give you a script that plots the Bayesian Linear Regression together with a few linear models with $w$ sampled from $p(w|D)$.

**QUESTIONS**:
- How should we interpret the two different sums in $\hat{\sigma}^2$?

In [ ]:
fig, ax = plt.subplots()
x_test = np.linspace(X_RANGE[0]-.5, X_RANGE[1]+.5, 100).reshape(-1, 1)
x_test = np.hstack((np.ones_like(x_test), x_test))

x_train, y_train = gen_data(f_type="linear")
x_train = _transform(x_train)
noise = 1e-6

Sn = x_train.T @ x_train
Sn += noise * np.eye(Sn.shape[0])
Sn_inv = np.linalg.inv(Sn)

# Closed-form posterior mean and variance
mu = (x_test @ Sn_inv @ x_train.T @ y_train).squeeze()
sig = x_test @ Sn_inv @ x_test.T          # full covariance over test points
di = np.sqrt(np.diag(sig) + noise)         # marginal std (+ noise for obs uncertainty)

# Sample w from the posterior p(w|D) = N(m_n, S_n^{-1})
m_n = (Sn_inv @ x_train.T @ y_train).squeeze()   # posterior mean of w, shape (2,)
n_samples = 15
rng = np.random.default_rng(42)
w_samples = rng.multivariate_normal(m_n, Sn_inv, size=n_samples)  # (n_samples, 2)

# Plot individual lines from sampled weights
for i, w in enumerate(w_samples):
    y_line = x_test @ w   # (100,)
    ax.plot(x_test[:, 1], y_line, color='steelblue', alpha=0.35,
            label=r'sampled $w$' if i == 0 else None)

# Plot closed-form mean ± 2 std
ax.plot(x_test[:, 1], mu, color="green", label='posterior mean', zorder=3)
ax.fill_between(x_test[:, 1], mu - 2*di, mu + 2*di,
                color="green", alpha=0.25, label=r'$\pm 2\sigma$ (closed form)')

# Plot training data on top
ax.plot(x_train[:, 1], y_train, 'o', color='red', label="data", zorder=4)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend()
plt.show()


---
# PART 2: Gaussian Processes
---
In part 1, we saw that when marginalizing over the weights $w$, we get that $p(y|D)$ follows a joint normal distribution. That gives us better uncertainty estimates, but it is still a linear model. We will now extend it in two ways to arrive a Gaussian Processes:
- We use the Woodbury matrix trick to get the dual formulation and then replace the inner products with the kernel
- We extend on the note that our prediction $p(y|D)$ is normal, to that $p(y)\sim N(\mu, \sigma^2)$, for some $\mu$ and $\sigma$, for any arbitrary set of $y$

All in all, we have that
$$p(y)\sim N(\mu, K),$$
where $K$ is the Gram matrix and $\mu$ is some mean function (that can depend on $x$). We will, as is standard, assume the $\mu = 0$. Hence,
$$p(y)\sim N(\mathbf{0}, K).$$

We can use this as a predictive tool through conditioning on observed data. Assume that $D=(X,y)$ and that we want to predict $y_*$ for feature values $x_*$, we split $K$ into $K_D$, $k^*$, and $k^**$, where $K_D$, contains the kernel function applied pairwise on the elements in $X$, $k^*$, contains the kernel values for pairs with one element in the $X$ and one in $x^{*}$ and $k^{**}$ is between points in $x^*$. I.e.,

$$
K = \begin{bmatrix}
K_D & k^* \\
k^{*\top} & k^{**}
\end{bmatrix},
$$
$$
k^* = (k(x_1, x^*), k(x_2, x^*), .., k(x_n, x^*)),
$$
$$
\text{and}~~ k^{**} = k(x^*, x^*),
$$
then, from the rules of conditioning for multivariate Gaussian distributions, we have that
$$
p(y^*|D)\sim \mathcal{N}(k^*K_D^{-1}y, k^{**} - k^*K_D^{-1}k^{*T})
$$

_Note_: The mean predictions is identical to what we say for Kernel Linear Regression, but now we additionally have well calibrated uncertainty estimates.



## Implementing your own GP

We will now use the mean and covariance/kernel functions to build a GP.
In the `__call__` function, we will compute the posterior distribution for a vector of points `x`. The output will __not__ be an array, but a __multivariate Gaussian__ with the dimensionality equal to the length of `x`.


In the `train`  function, we will set the training data and calculate a critical component: the inverse of the covariance matrix $K^{-1}$.
Usually, this is done using Cholesky decomposition for numerical stability, but we will use `np.linalg.inv` to calculate the inverse.

Pro tip: You might run into problems when computing the inverse due to numerical instabilities. If this happens, add a value of $10^{-6}$ to the diagonal elements of the matrix before computing the inverse.

**Your task:** Compute the covariance matrix $K$ and its inverse in `__init__` and calculate the posterior distribution in `__call__`.

In [ ]:
class GaussianProcess:
    def __init__(
            self,
            x_train: np.ndarray,
            y_train: np.ndarray,
            kern_function: Callable[[np.ndarray, np.ndarray, float], np.ndarray],
            length_scale: float,
            noise: float,
    ):
        """We initialize the GP with the lengthscale and noise parameters + the training data.
        """

        # setting the kernel function, fix the length scale so we don't have to pass it all the time
        self.x_train: np.ndarray = x_train
        self.y_train: np.ndarray = y_train
        self.kern_fn = kern_function
        self.length_scale = length_scale
        self.kern = lambda x, y: kern_function(x, y, length_scale)
        self.noise = noise

        # some checks
        assert x_train.ndim == 2 and y_train.ndim == 2, 'x_train and y_train should be 2D arrays'
        assert x_train.shape[0] == y_train.shape[0], 'first dimension of x_train and y_train has to be equal (n_points)'
        assert y_train.shape[1] == 1, 'y_train has to be of form (n_points, 1)'


        # Due to numerical instabilities, the covariance matrix might not be invertible.
        # We add a small constant value to the diagonal elements ('jitter') to enforce the
        # matrix to be positive semidefinite (which implies invertibility)
        # See, e.g., https://scicomp.stackexchange.com/questions/36342/advantage-of-diagonal-jitter-for-numerical-stability
        #
        #
        # The inputs x_train and y_train both have to be 2D here. x_train has shape (n_points, dim_of_points), y_train has
        # shape (n_points, 1)
        #
        # ➡️ TODO : compute the covariance and the inverse of the covariance matrix here. Save them as class attributes so we don't need to
        #    TODO : recompute them all the time ⬅️

        self.K = ...
        self.K_inv = ...


    def __call__(self, x: np.ndarray) -> sp.stats._multivariate.multivariate_normal_frozen:

        assert self.x_train is not None and self.y_train is not None, "Have to train the GP before calling"
        if x.ndim == 1:
            x = x.reshape(-1, 1)

        # First, compute the posterior mean and covariance (use self.K_inv and the definitions above),
        # then compute the posterior distribution which is a multivariate normal distribution
        # Hint: you might need to add a small diagonal value to the covariance matrix again
        # Use the code from above for that (don't add more than 1e-6)
        # ➡️ TODO : compute the posterior distribution ⬅️

        return ...


    def posterior_mean(self, x: np.ndarray) -> np.ndarray:
        """Convenience function to extract the mean of the posterior"""
        if x.ndim == 1:
            x = x.reshape(-1, 1)
        dist = self(x)
        return dist.mean

    def posterior_covariance(self, x: np.ndarray) -> np.ndarray:
        """Convenience function to extract the covariance of the posterior"""
        if x.ndim == 1:
            x = x.reshape(-1, 1)
        dist = self(x)
        return dist.cov

    def log_marginal_likelihood(
            self,
    ) -> float:

        # ➡️ TODO : Implement this when you're instructed to in the notebook, you can skip this for now otherwise ⬅️
        return ...



**Safety check:** Run the following cell to make sure that everything is working correctly.

In [ ]:
x_check_train = np.random.rand(5, 1)
y_check_train = np.random.rand(5, 1)
x_check_test = np.random.rand(3, 1)
gp = GaussianProcess(x_check_train, y_check_train, kern, 2, 0.1)
pred = gp(x_check_test)
if isinstance(pred, sp.stats._multivariate.multivariate_normal_frozen):
    print("✅ All good.")
else:
    print("🚨 __call__ does not return a multivariate distribution. Make sure to use 'sp.stats.multivariate_normal'")

## Testing our implementation

Now that we have our GP implemented, let's take it for a spin.

To see how it behaves, we provide a small plotting script that plots:
- The predicted mean
- The 95% confidence interval
- A few samples drawn from the distribution of possible functions

In [ ]:
def plot_gp(
        gp: GaussianProcess,
        x_test: np.ndarray,
        ax: plt.Axes = None
) -> None:
    """
    Plot the GP posterior distribution of gp for a given set of test points (x_test and y_test).
    Accepts an optional parameter ax which plots it on an existing ax, otherwise a new figure
    is created.
    """
    if ax is None:
        fig, ax = plt.subplots(1, 1)
    posterior_distribution = gp(x_test)
    rvs = posterior_distribution.rvs(10)

    ci_lb = []
    ci_ub = []

    for _x in x_test:
        x_marginal = gp(_x)
        _mean = x_marginal.mean.squeeze()
        variance = x_marginal.cov.squeeze()

        _lb, _ub = sp.stats.norm.interval(0.95, loc=_mean, scale=np.sqrt(variance))
        ci_lb.append(_lb)
        ci_ub.append(_ub)

    ci_lb = np.array(ci_lb)
    ci_ub = np.array(ci_ub)

    for i, rv in enumerate(rvs):
        ax.plot(x_test.squeeze(), rv, alpha=0.5, color='blue', label='posterior sample' if i == 0 else None)
    ax.fill_between(x_test.squeeze(), ci_lb, ci_ub, color='gray', alpha=0.5, label='95% CI')
    ax.scatter(gp.x_train, gp.y_train, marker='x', color='red', label='training data', zorder=10)
    ax.set_ylabel('y(x)')
    ax.legend()

**Task**: Initialise a GP with a reasonable lengthscale and noise level

Here, we reuse our previous data, but with fewer samples, as it makes it easier to see what is going on.

In [ ]:
x_train, y_train = gen_data(f_type="sinus", n=11)
x_test = np.linspace(X_RANGE[0], X_RANGE[1], 100).reshape(-1, 1)


# ➡️ TODO : Create a new GP instance with a lengthscale of your choice and plot it.
#    TODO : Try different lengthscales to see how the affect the behavior of the GP.  ⬅️
gp = ...


plot_gp(gp, x_test)
plt.show()

## Systematic approach to fit lengthscales: MLE

We saw above that setting the lengthscale incorrectly can render the Gaussian process useless: if it is set too low or too high, the Gaussian process fails in modelling the function reasonably (if you didn't see that, try lengthscales of 0.1 and 10).
While you probably found a lengthscale value that led to reasonable performance, we want to set the lengthscale automatically because what a "good value" is depends on the function at hand.
All approaches to set the lengthscale (and often more hyperparameters) are in some form **maximizing the marginal likelihood of seeing the training data under the GP prior** (i.e., we will optimize our lengthscale with MLE).


Note that $\mathbf{y}$ is a $n$-dimensional vector which represents one possible realization of the underlying true function at the locations $X \in \mathbb{R}^{n\times d}$.
The vector $\mathbf{y}|X$ follows a multivariate normal distribution, and as such:
$$
\hat{l} = \arg\max_l p(\mathbf{y}|X,l)
$$

We state the posterior (log) marginal likelihood here and refer to [Gaussian Processes for Machine Learning](https://gaussianprocess.org/gpml/chapters/RW.pdf) for a derivation:
$$
\log p(\mathbf{y}|X,l) = -\frac{1}{2}\mathbf{y}^\intercal K^{-1}\mathbf{y}-\frac{1}{2}\log |K| -\frac{n}{2}\log 2\pi.
$$


**YOUR TASK**: Implement the `log_marginal_likelihood` in the `GP` class above.

### Maximize the marginal likelihood by minimizing the negative marginal log likelihood

We optimize the negative_marginal_log_likelihood over the log of the lengthscale and the log of the noise instead of directly over the parameter.

**Question:** What is the benefit of doing that?

**Safety check:** I get a lengthscale close to `[0.0886]` and noise close to `[0.0092]` for the "Sinus" data set with `n=11`.

In [ ]:
def fit_gp(gp: GaussianProcess) -> GaussianProcess:
    """Fits the hyperparameters of a GP by maximizing the marginal likelihood.
    Returns a new GaussianProcess with the optimized lengthscale and noise."""

    # Define an objective for scipy
    def negative_marginal_log_likelihood(params: np.ndarray) -> float:
        length_scale = np.exp(params[0])
        noise_scale = np.exp(params[1]) + 1e-5   # adding a minimum noise to make K invertible
        candidate = GaussianProcess(gp.x_train, gp.y_train, gp.kern_fn, length_scale, noise_scale)
        return -candidate.log_marginal_likelihood()

    # Use scipy minimize to optimize log-ls and log-noise
    params = sp.optimize.minimize(negative_marginal_log_likelihood, x0=np.array([np.log(0.1), np.log(1e-2)]))['x']
    best_ls = np.exp(params[0])
    best_noise = np.exp(params[1]) + 1e-6  # adding a minimum noise to make K invertible
    return GaussianProcess(gp.x_train, gp.y_train, gp.kern_fn, best_ls, best_noise)


gp_init = GaussianProcess(x_train, y_train, kern, 0.3, 1e-2)
gp_fitted = fit_gp(gp_init)
print(f"best lengthscale: {gp_fitted.length_scale:.4f}, best noise: {gp_fitted.noise:.3e}")

### Plot the GP with lengthscale fitted by MLE

In [ ]:
plot_gp(
    gp = gp_fitted,
    x_test = x_test,
    ax = None,
)
plt.show()

# PART 3: Bayesian Optimization

Bayesian optimization is a technique for efficiently finding the min/max value of an underlying hidden function by iteratively evaluating the function. The idea is to use our current knowledge about the function to select interesting points to query next.

Up until this point, we have worked with pointwise data sets $D = (X, y)$. Now we switch to assuming some underlying function $f(x)$, where we have observed some values $D = (X, f(x) + \varepsilon)$, potentially with noise.

The general strategy of BO is to first select a few points through random sampling and the iteratively fit a Gaussian process on the current data set, and based on that model select the next point to evaluate.


In [ ]:
def fn(
        x: np.ndarray
) -> np.ndarray:
    y = -np.sin(10 * x) + np.sin(30 * x) + x
    return y

# define lower and upper bounds of the function: we will only consider the function within these bounds
lb, ub = 0, 1

initial_x = np.random.default_rng(seed=567).random(5).reshape(-1, 1)
initial_y = fn(initial_x)

plt.plot(np.linspace(lb, ub, 200), fn(np.linspace(lb, ub, 200)), label='f(x)')
plt.scatter(initial_x, initial_y, color='red', label='observations', zorder = 4)
plt.legend()
plt.show()

## Acquisition functions

New points are selected by maximizing a so-called _Acquisition function_.
While it may seem strange to solve an optimization problem (maximizing the acquisition function) to maximize our black-box function, the acquisition function can be maximized using gradient-based approaches because it has a closed-form expression.

We will work with a popular acquisition function: __Expected improvement (EI)__.
EI describes by how much, in expectation w.r.t. to our GP posterior after $n$ observations, a given point improves over current best function value observed so far $y^*_n$:
$$
EI_n(x) = \mathbb{E}_{y\sim GP(X,\mathbf{y})}\left[[y(x)-y^*_n]^+ \right]
$$
Note that $[x]^+:=\max(0,x)$.

EI naturally does an _exploration-exploitation tradeoff_: points that have already been evaluated get zero EI (in noiseless models as in this lab) but regions where all possible functions have a value worse than the current best point also get zero EI.
The sweet spot are regions that are under-explored but are also promising.

Since our GP posterior follows a multivariate normal distribution, we can find a [closed form expression for the expected improvement](https://ekamperi.github.io/machine%20learning/2021/06/11/acquisition-functions.html). Let the mean and variance of our new point $(x,y)$ be $\mu(x)$ and $\sigma(x)$, as given by the GP. Then:
$$
EI_n(x) = \sigma(x)\varphi (Z(x)) + \sigma(x)Z(x)\Phi (Z(x)),
$$
where $Z(x) = \frac{\mu_n(x)-y^*_n}{\sigma}$ is the expected difference between the point $x$ and the best function values after $n$ observations. Further, $\varphi(x)$ is the probability density function of the *standard* normal distribution, and $\Phi$ is the cummulative density function of the *standard* normal distribution.

__YOUR TASK:__ Implement the expected improvement acquisition function by adding missing lines in the cell below. You can use sp.stats for pds and cdfs.

In [ ]:
def expected_improvement(
    gp: GaussianProcess,
    x: np.ndarray,
    best_observed_value: float
) -> np.ndarray:

    if x.ndim == 1:
        x = x.reshape(-1, 1)
    eis = []
    for _x in x:
        # ➡️ TODO : Implement the expected improvement acquisition function  ⬅️
        ei = ...

        eis.append(ei)
    return np.array(eis)

__Safety check:__ (make sure you executed all cells in order up to here, i.e., you used the GP with MLE).

You should get the following figure:

<img src="https://github.com/ErikOrm/EDAP30-Lab1/blob/main/EI.png?raw=1" alt="drawing" width="400"/>

In [ ]:
gp = fit_gp(GaussianProcess(initial_x, initial_y, kern, 0.3, 1e-4))
plt.plot(expected_improvement(gp, np.linspace(lb, ub, 300).reshape(-1, 1), initial_y.max()).squeeze(), label=rf'$EI_n(x)$')
plt.legend()
plt.show()

Now, we have all the ingredients to do Bayesian optimization. We will
* Sample some initial points and evaluate them
* Fit a GP model with MLE
* Find the next point to evaluate by maximizing the Expected improvement
* Evaluate the next point, add it to the data and start over from the second point

We will now see how all components play together.

**Question (after running the BO loop):** Why does the ei graph become sharper and sharper for every passing iteration?

In [ ]:
x_train = initial_x
y_train = initial_y


gp = fit_gp(GaussianProcess(x_train, y_train, kern, 0.3, 1e-2))
ei = lambda x: expected_improvement(gp, x, y_train.max())

for i in range(10):
    print('###########################')
    print(f"####### iteration {i} #######")
    print('###########################')

    fig, axs = plt.subplots(2, 1, sharex=True, figsize=(7, 7))
    # plot current GP
    plot_gp(gp, x_test, ax=axs[0])
    # stupidly optimize the acquisition function
    x_range = np.linspace(lb, ub, 1000)[:, np.newaxis]
    eis = ei(x_range)
    x_next = x_range[np.argmax(eis)]

    # plot acquisition function
    axs[1].plot(x_range, eis)
    axs[1].set_ylabel('EI(x)')
    axs[1].vlines(x_next, ymin=ei(x_next), ymax=0, color='green', linestyle='dashed', label='best x')
    axs[1].legend()
    plt.show(fig)
    # evaluate point where acquisition function maximum
    y_next = fn(x_next)
    # add to training data
    x_train = np.vstack((x_train, x_next[np.newaxis, :]))
    y_train = np.vstack((y_train, y_next[np.newaxis, :]))

    # retrain the GP and update the EI
    gp = fit_gp(GaussianProcess(x_train, y_train, kern, 0.3, 1e-4))
    ei = lambda x: expected_improvement(gp, x, y_train.max())
